In [1]:
import pandas as pd

# ============================================================
# 1. DADOS DA ACADEMIA
# ============================================================
planos = {
    "Mensal": {"preco":  423, "alunos": 40},      # R$ 423 por mês, por aluno
    "Trimestral": {"preco": 1200, "alunos": 30},  # R$ 1200 a cada 3 meses ou 400 ao mês, por aluno
    "Semestral": {"preco": 2286, "alunos": 15},   # R$ 2286 a cada 6 meses ou 381 ao mês, por aluno
    "Anual": {"preco": 4308, "alunos": 15}        # R$ 4308 por ano ou 359 ao mês, por aluno
}

# Custos fixos mensais
custos_fixos = {
    "Aluguel": 5000,
    "Internet": 250,
    "Energia": 350,
    "Água": 180,
    "Manutenção": 400,
    "Marketing": 500
}

# ============================================================
# Parâmetros de divisão de pró-labore (devem somar 100%)
# Ex.: 0.6 + 0.4 = 1.0  (60% / 40%)
# ============================================================
part1 = 0.50
part2 = 0.30
part3 = 0.20

socios = {
    "Sócio-1 (50%)": part1,
    "Sócio-2 (30%)": part2,
    "Sócio-3 (20%)": part3
}

assert abs(sum(socios.values()) - 1.0) < 1e-9, "As porcentagens dos sócios devem somar 100%."

# ============================================================
# 2. CÁLCULO DA RECEITA
# ============================================================
receita_dados = []
for plano, dados in planos.items():
    if plano == "Semestral":
       receita_mensal = (dados["preco"] / 6) * dados["alunos"] # distribui por mês
       explicar_seme = ((receita_mensal / dados["alunos"]))
       qt_alunos_4 = dados["alunos"]  
    
                          
    elif plano == "Trimestral":
        receita_mensal = (dados["preco"] / 3) * dados["alunos"]  # distribui por mês
        explicar_trim = ((receita_mensal / dados["alunos"]))
        qt_alunos_2 = dados["alunos"]
        
    elif plano == "Anual":
        receita_mensal = (dados["preco"] / 12) * dados["alunos"] # distribui por mês
        explicar_ano = ((receita_mensal / dados["alunos"]))
        qt_alunos_3 = dados["alunos"]
        
    else:
        receita_mensal = dados["preco"] * dados["alunos"]
        explicar_mes = ((receita_mensal / dados["alunos"]))
        qt_alunos_1 = dados["alunos"]
        
    receita_dados.append([plano, dados["alunos"], dados["preco"], receita_mensal])
    

df_receita = pd.DataFrame(
    receita_dados,
    columns=["Plano", "Qtde Alunos", "Contrato(R$)", "Receita Mensal (R$)"]
)

receita_total = df_receita["Receita Mensal (R$)"].sum()

# ============================================================
# 3. CÁLCULO DOS CUSTOS
# ============================================================
df_custos = pd.DataFrame(list(custos_fixos.items()), columns=["Custo", "Valor (R$)"])
custo_total = df_custos["Valor (R$)"].sum()

# Tributos (simplificado)
tot_tributos = receita_total * 0.06

# ============================================================
# 4. PRÓ-LABORE (simplificado)
# Base: 40% do lucro operacional (receita - custos fixos), menos INSS (11%) e IRPF (14%)
# ============================================================
base_prolab = max(((receita_total - custo_total)) * 0.40, 0)  # evita negativo

desc_inss = base_prolab * 0.11
desc_irpf = base_prolab * 0.14
val_prolab_liq = ((base_prolab - desc_inss - desc_irpf) + (receita_total/5)) # 100% provisionado para prolabore

# Distribuição entre sócios
dist_prolab = {nome: val_prolab_liq * pct for nome, pct in socios.items()}
tot_prolab = sum(dist_prolab.values())

# ============================================================
# 5. RESULTADO FINAL
# ============================================================
lucro_liquido = (receita_total - custo_total) - (tot_prolab + tot_tributos)
perc_liq = (lucro_liquido / receita_total) * 100 if receita_total else 0.0
despesa_total = receita_total - lucro_liquido     # = custos + tributos + pró-labores
perc_despesa_total = (despesa_total / receita_total) * 100 if receita_total else 0.0
pd.options.display.float_format = "{:,.2f}".format #opção global de exibição p/ valores com 2 casas decimais

#================================================================
# Imprimindo os resultados
#================================================================
print("="*80)
print(" ACADEMIAS - RELATÓRIO FINANCEIRO RESUMIDO")
print("="*80)

print("\n--- Receita por Plano ---")
print(df_receita.to_string(index=False))

#================================================================
# Explicação do Relatório Financeiro
#================================================================

#=================
# Visão Trimestral
#=================
fat_mes = ((explicar_mes * qt_alunos_1) * 3) 
fat_trim = ((explicar_trim * qt_alunos_2) * 3)
fat_seme = ((explicar_seme * qt_alunos_4) * 3)
fat_ano = ((explicar_ano * qt_alunos_3) * 3)
total_trimestre = (fat_mes+fat_trim+fat_seme+fat_ano)

#=================
# Visão Anual
#=================
demo_mes = ((explicar_mes * qt_alunos_1) * 12)
demo_trim = ((explicar_trim * qt_alunos_2) * 12)
demo_seme = ((explicar_seme * qt_alunos_4) * 12)
demo_ano = ((explicar_ano * qt_alunos_3) * 12)
demo_rec_tot = (demo_mes+demo_trim+demo_seme+demo_ano)

#===========================================================
# Rotina de impressão - Detalhamento do relatório financeiro
#===========================================================
print("")
print("=" * 80)
print("ACADEMIAS - DETALHAMENTO DO RELATÓRIO FINANCEIRO")
print("=" * 80)
#===========================================================
print(f"Mensal:     Val_Aula =  {explicar_mes:.2f}  | Trimestre =  {fat_mes:.2f}  | Fatura/Mes = {explicar_mes * qt_alunos_1:.2f}") 
print(f"Trimestre:  Val_Aula =  {explicar_trim:.2f}  | Trimestre =  {fat_trim:.2f}  | Fatura/Mes = {explicar_trim * qt_alunos_2:.2f}") 
print(f"Semestre:   Val_Aula =  {explicar_seme:.2f}  | Trimestre =  {fat_seme:.2f}  | Fatura/Mes =  {explicar_seme * qt_alunos_4:.2f}") 
print(f"Anual:      Val_Aula =  {explicar_ano:.2f}  | Trimestre =  {fat_ano:.2f}  | Fatura/Mes =  {explicar_ano * qt_alunos_3:.2f}")       
print("="* 80)

#===========================================================
# Rotina de impressão - Totalizações
#===========================================================
print(f"Fat_Trimestral: -------------------------> R$ {total_trimestre:.2f}  | Anual    R$ {demo_rec_tot:.2f}")
print(" ")

print("=" * 80)
print("DEMONSTRATIVO DOS RESULTADOS")
print("=" * 80)

print(f"Receita Total Mensal: R$ {receita_total:,.2f}")
print(f"Custos Fixos (Despesas Permanentes): R$ {custo_total:,.2f}")
print(f"Tributos (6% s/ receita): R$ {tot_tributos:,.2f}")
print("\n Pró-Labore                  Rubricas")

for nome, valor in dist_prolab.items():
    print(f"{nome:<14}              R$ {valor:,.2f}")

print(f"Somatório dos Pró-labores: R$ {tot_prolab:,.2f}")

#===========================================================
# Rotina de impressão - Demonstração dos resultados
#===========================================================
print("\n--- Despesas & Resultado ---")
print(f"Despesas Totais Mensais: R$ {despesa_total:,.2f}")
print(f"Perc. Despesa Total: {perc_despesa_total:,.2f}%")
print("="*80)
print(f">>> Lucro Líquido Mensal: R$ {lucro_liquido:,.2f}")
print(f">>> Lucro Líquido: {perc_liq:,.2f}% da Receita Total Mensal")
print("="*80)



 ACADEMIAS - RELATÓRIO FINANCEIRO RESUMIDO

--- Receita por Plano ---
     Plano  Qtde Alunos  Contrato(R$)  Receita Mensal (R$)
    Mensal           40           423            16,920.00
Trimestral           30          1200            12,000.00
 Semestral           15          2286             5,715.00
     Anual           15          4308             5,385.00

ACADEMIAS - DETALHAMENTO DO RELATÓRIO FINANCEIRO
Mensal:     Val_Aula =  423.00  | Trimestre =  50760.00  | Fatura/Mes = 16920.00
Trimestre:  Val_Aula =  400.00  | Trimestre =  36000.00  | Fatura/Mes = 12000.00
Semestre:   Val_Aula =  381.00  | Trimestre =  17145.00  | Fatura/Mes =  5715.00
Anual:      Val_Aula =  359.00  | Trimestre =  16155.00  | Fatura/Mes =  5385.00
Fat_Trimestral: -------------------------> R$ 120060.00  | Anual    R$ 480240.00
 
DEMONSTRATIVO DOS RESULTADOS
Receita Total Mensal: R$ 40,020.00
Custos Fixos (Despesas Permanentes): R$ 6,680.00
Tributos (6% s/ receita): R$ 2,401.20

 Pró-Labore               